# Databricks/PySpark for Data Analysts/Engineers — Module 2: DataFrame API Basics

In this module we walk through the fundamental operations on DataFrames: how to create
them, how to select and transform columns, how to filter rows, and how to sort the
result. If you know `pandas` or SQL from earlier courses, most of this will look
familiar -- deliberately so.

## Table of contents
1. [Creating DataFrames](#sec1)
2. [Explicit schemas](#sec2)
3. [Selecting and transforming columns](#sec3)
4. [Filtering rows](#sec4)
5. [Adding and dropping columns](#sec5)
6. [Sorting](#sec6)
7. [Converting to/from pandas](#sec7)
8. [Summary and exercises](#sec8)


We start (or grab the already-running) `SparkSession` -- the entry point to everything in Spark. In Databricks, `spark` is already available automatically in every notebook; the same code also works locally.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("spark_course").getOrCreate()
spark


Let's load our shared dataset -- online store data (`customers`, `products`, `employees`, `orders`, `order_items`), the exact same schema as the SQL course, but this time as Spark DataFrames.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- customers -----------------------------------------------------------
n_customers = 300
cities = ["Warsaw", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segments = ["New", "Standard", "Premium"]
segment_probs = [0.35, 0.45, 0.20]

customers_pd = pd.DataFrame({
    "customer_id": np.arange(1, n_customers + 1),
    "city": rng.choice(cities, size=n_customers, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segments, size=n_customers, p=segment_probs),
    "days_since_signup": rng.integers(1, 900, size=n_customers),
})

# --- products ------------------------------------------------------------
n_products = 40
categories = ["Electronics", "Clothing", "Home", "Sports", "Books"]
avg_prices = {"Electronics": 1200, "Clothing": 150, "Home": 250, "Sports": 300, "Books": 50}

product_categories = rng.choice(categories, size=n_products)
products_pd = pd.DataFrame({
    "product_id": np.arange(1, n_products + 1),
    "category": product_categories,
    "price": [
        round(max(5.0, rng.normal(avg_prices[c], avg_prices[c] * 0.3)), 2)
        for c in product_categories
    ],
})

# --- employees ----------------------------------------------------------
n_employees = 12
regions = ["North", "South", "East", "West"]
employees_pd = pd.DataFrame({
    "employee_id": np.arange(1, n_employees + 1),
    "region": rng.choice(regions, size=n_employees),
    "department": rng.choice(["Sales", "Support"], size=n_employees, p=[0.6, 0.4]),
})

# --- orders ------------------------------------------------------------
n_orders = 3000
order_customer_id = rng.choice(customers_pd["customer_id"], size=n_orders)
order_employee_id = rng.choice(employees_pd["employee_id"], size=n_orders)

segment_map = customers_pd.set_index("customer_id")["segment"]
order_segment = pd.Series(order_customer_id).map(segment_map).values

# cancellation probability depends on customer segment (deliberately built in)
cancel_prob = np.where(
    order_segment == "New", 0.22,
    np.where(order_segment == "Standard", 0.10, 0.04),
)
cancelled = rng.binomial(1, cancel_prob)
order_status = np.where(
    cancelled == 1, "Cancelled",
    rng.choice(["Placed", "Shipped", "Delivered"], size=n_orders, p=[0.2, 0.3, 0.5]),
)

base_date = pd.Timestamp("2024-01-01")
day_offset = rng.integers(0, 730, size=n_orders)

orders_pd = pd.DataFrame({
    "order_id": np.arange(1, n_orders + 1),
    "customer_id": order_customer_id,
    "employee_id": order_employee_id,
    "order_date": [base_date + pd.Timedelta(days=int(d)) for d in day_offset],
    "status": order_status,
})

# --- order_items ------------------------------------------------------
item_rows = []
item_id = 1
product_price = products_pd.set_index("product_id")["price"]

for order_id in orders_pd["order_id"]:
    n_items = rng.integers(1, 5)
    chosen_products = rng.choice(products_pd["product_id"], size=n_items, replace=False)
    for product_id in chosen_products:
        quantity = int(rng.integers(1, 4))
        discount = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        item_rows.append({
            "item_id": item_id,
            "order_id": int(order_id),
            "product_id": int(product_id),
            "quantity": quantity,
            "unit_price": float(product_price[product_id]),
            "discount": float(discount),
        })
        item_id += 1

order_items_pd = pd.DataFrame(item_rows)

# --- convert to Spark DataFrames ---------------------------------------
customers = spark.createDataFrame(customers_pd)
products = spark.createDataFrame(products_pd)
employees = spark.createDataFrame(employees_pd)
orders = spark.createDataFrame(orders_pd)
order_items = spark.createDataFrame(order_items_pd)

customers.show(5)


<a id="sec1"></a>
## 1. Creating DataFrames

A Spark DataFrame can be created in several ways -- we already saw one
(`createDataFrame` from a pandas DataFrame in Module 1). Here are the most common:

In [ ]:
# From a list of tuples + column names
small_df = spark.createDataFrame(
    [(1, "Mug", 19.99), (2, "Plate", 12.50)],
    ["id", "name", "price"],
)
small_df.show()


In [ ]:
# spark.range() -- a quick way to generate a sequence of numbers (useful for testing)
range_df = spark.range(5)
range_df.show()


> ⚡ **createDataFrame from pandas vs. reading a file**
>
> In Module 1 we created our data with `spark.createDataFrame(pandas_df)` -- convenient for learning and small datasets. In real work, data is most often loaded directly from files (`spark.read.csv(...)`, `spark.read.parquet(...)`) -- we'll cover that in detail in Module 6.

<a id="sec2"></a>
## 2. Explicit schemas

When you create a DataFrame from raw data (e.g. a CSV file with no type header), Spark
might guess column types incorrectly. You can specify a schema EXPLICITLY with
`StructType`:

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("id", IntegerType(), nullable=False),
    StructField("name", StringType(), nullable=True),
    StructField("price", DoubleType(), nullable=True),
])

products_explicit_schema = spark.createDataFrame(
    [(1, "Mug", 19.99), (2, "Plate", 12.50)], schema=schema
)
products_explicit_schema.printSchema()


> ⚠️ **Schema inference (inferSchema) has a cost**
>
> When Spark guesses types itself (e.g. loading CSV with `inferSchema=True`), it has to READ the data twice -- once to guess the types, once to actually load it. On large datasets that's a noticeable cost. An explicit schema avoids this double read -- we'll come back to this in Module 6.

<a id="sec3"></a>
## 3. Selecting and transforming columns

`select()` picks a subset of columns -- exactly like `SELECT` in SQL. Columns can be
referenced in a few equivalent ways: by name as a quoted string, via `df.column_name`,
or via `F.col("column_name")` (the last one is the most flexible, since it lets you
build expressions).

In [ ]:
from pyspark.sql import functions as F

customers.select("customer_id", "city").show(5)


In [ ]:
# A column expression + alias -- create a new, computed column on the fly
customers.select(
    "customer_id",
    F.col("city"),
    (F.col("days_since_signup") / 365).alias("years_as_customer"),
).show(5)


> ⚡ **col() vs. selectExpr() vs. plain SQL**
>
> The same transformation can be written a few ways: `F.col("x") + 1`, or `df.selectExpr("x + 1 AS y")` (SQL-like syntax, as one string), or plain Spark SQL entirely (Module 5). They all compile down to the same execution plan -- pick whichever reads most naturally in a given spot.

<a id="sec4"></a>
## 4. Filtering rows

`filter()` (or its alias `where()`) corresponds to SQL's `WHERE`. Conditions are combined
with `&` (and), `|` (or), `~` (not) -- NOTE: not plain Python `and`/`or`/`not`, since
those operate on single boolean values, not on whole columns.

In [ ]:
customers.filter(F.col("segment") == "Premium").show(5)


In [ ]:
# A compound condition -- remember the parentheses around each sub-condition!
customers.filter((F.col("segment") == "Premium") & (F.col("city") == "Warsaw")).show(5)


> ⚠️ **Parentheses are mandatory in compound conditions**
>
> `F.col("a") == 1 & F.col("b") == 2` without parentheses will behave incorrectly (or not run at all) -- Python resolves `&`/`|` operator precedence differently from what you'd intuitively expect. Always wrap EVERY sub-condition in its own parentheses, as in the example above.

<a id="sec5"></a>
## 5. Adding and dropping columns

`withColumn()` adds a new column (or OVERWRITES an existing one, if you give it the same
name). `withColumnRenamed()` renames a column. `drop()` removes a column.

In [ ]:
orders_with_flag = orders.withColumn(
    "is_cancelled", F.col("status") == "Cancelled"
)
orders_with_flag.show(5)


In [ ]:
orders_renamed = orders.withColumnRenamed("order_date", "date")
orders_without_employee = orders.drop("employee_id")
orders_without_employee.printSchema()


<a id="sec6"></a>
## 6. Sorting

`orderBy()` (or `sort()`, they're synonyms) sorts rows -- ascending by default, `.desc()`
on a column reverses the direction.

In [ ]:
products.orderBy(F.col("price").desc()).show(5)


In [ ]:
# Sorting by multiple columns at once
products.orderBy(F.col("category"), F.col("price").desc()).show(10)


<a id="sec7"></a>
## 7. Converting to/from pandas

`.toPandas()` pulls the ENTIRE Spark DataFrame into the Driver's memory as a plain
`pandas.DataFrame` -- useful at the end of an analysis (e.g. to draw a `matplotlib`
chart), but dangerous with large data.

In [ ]:
products_pd = products.limit(5).toPandas()
print(type(products_pd))
products_pd


> ⚠️ **toPandas() on a large DataFrame can flood the Driver's memory**
>
> `.toPandas()` pulls ALL rows onto one machine (the Driver) -- if the DataFrame has a billion rows, it probably won't fit in the Driver's memory and the application will crash. Always REDUCE the data in Spark first (filter, aggregate, `.limit()`) and only convert the (now small) RESULT to pandas.

<a id="sec8"></a>
## 8. Summary and exercises

In this module we covered:
- several ways to create DataFrames (`createDataFrame`, `spark.range()`),
- defining an explicit schema with `StructType`/`StructField`,
- selecting and transforming columns (`select`, `F.col()`, aliases),
- filtering rows (`filter`/`where`, the `&`/`|`/`~` operators),
- adding, renaming, and dropping columns (`withColumn`, `withColumnRenamed`, `drop`),
- sorting (`orderBy`),
- converting to/from `pandas` (`.toPandas()`) and its memory trap.

### 📝 Exercise 1: Selecting and transforming columns

From the `products` DataFrame, select the `product_id`, `category` columns and a new column `price_with_vat` (price times `1.23`).

<details>
<summary>Show solution</summary>

```python
products.select(
    "product_id",
    "category",
    (F.col("price") * 1.23).alias("price_with_vat"),
).show(5)
```

</details>

### 📝 Exercise 2: Filtering orders

Filter `orders` down to those with status `'Delivered'` OR `'Shipped'`, placed after `2025-01-01`. Hint: comparing a date column with `F.col("order_date") > "2025-01-01"` works correctly on a date-typed column.

<details>
<summary>Show solution</summary>

```python
orders.filter(
    ((F.col("status") == "Delivered") | (F.col("status") == "Shipped"))
    & (F.col("order_date") > "2025-01-01")
).show(10)
```

</details>

### 📝 Exercise 3: A new column with a condition

Add a column `item_value` to `order_items`, computed as `quantity * unit_price * (1 - discount)`, then sort descending by that column and show the 5 most expensive items.

<details>
<summary>Show solution</summary>

```python
items_with_value = order_items.withColumn(
    "item_value", F.col("quantity") * F.col("unit_price") * (1 - F.col("discount"))
)
items_with_value.orderBy(F.col("item_value").desc()).show(5)
```

</details>

### 📝 Exercise 4: How many columns, how many rows

For the `employees` DataFrame, print the number of columns (`len(df.columns)`) and the number of rows (`count()`), without converting to pandas.

<details>
<summary>Show solution</summary>

```python
print(f"Number of columns: {len(employees.columns)}")
print(f"Number of rows: {employees.count()}")
```

</details>

> 🔥 **Challenge: your own schema and validation**
>
> Create a DataFrame `test_customers` with 3 rows of data, where ONE `customer_id` value is DELIBERATELY a string (`"abc"`) instead of a number, using an explicit `StructType` with `IntegerType()` for `customer_id`. Check what happens -- does Spark raise an error immediately, or only at an action?

<details>
<summary>Show solution</summary>

```python
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

test_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("city", StringType(), True),
])

try:
    test_customers = spark.createDataFrame(
        [(1, "Warsaw"), ("abc", "Krakow"), (3, "Gdansk")], schema=test_schema
    )
    test_customers.show()
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")
```

createDataFrame() itself is a transformation (really, a plan-building step) -- depending on the Spark version and whether the data is already materialized locally, the type-validation error can surface either right away at creation, or only at the first action (show()). This is a good illustration of the lazy evaluation from Module 1: errors can be 'lazy' too.

</details>

## What's next?

In **Module 3** we'll extend transformations with aggregations: `groupBy`, built-in
functions (`pyspark.sql.functions`), handling `NULL` values, and conditional
expressions with `when`/`otherwise`.